# Fine-tuning Detectree2 on Pléiades Neo RGB imagery

This notebook trains three leave-one-site-out Detectree2 models using 0.3 m Pléiades Neo imagery from Lokoué, Dzanga and Mbeli. Each model is trained using two sites, while the third is excluded from model development and retained for independent evaluation.

## Setup

Install fixed Detectron2 and Detectree2 revisions to stabilise the computational environment. Restart the Colab runtime after installation before continuing.

In [ ]:
DETECTRON2_COMMIT = "a2f4a8771ab77e8411c26b27f24f9489a28a2453"
DETECTREE2_COMMIT = "d9fb07f0dfb493f34def563c1ff896fecd59210d"

!pip -q install "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip -q install "git+https://github.com/PatBall1/detectree2.git@{DETECTREE2_COMMIT}"
!pip -q install rasterio geopandas

## Paths and configuration

Source imagery and annotations are read from Google Drive. Temporary H1 rasters and tiles are written to a dedicated local directory, while trained models are retained in `runs_rgb_v2`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import inspect
import json
import os
import shutil
import urllib.request

import geopandas as gpd
import numpy as np
import rasterio
import torch

ROOT = Path("/content/drive/MyDrive/Congo basin/congo")
WORK = Path("/content/h1_work")

STACKS = WORK / "stacks"
CROPS = WORK / "crops"
TILES = WORK / "tiles"
KEEP = WORK / "keep"
RUNS = ROOT / "runs_rgb_v2"

SITES = ["lokoue", "dzanga", "mbeli"]
STRATA = ["tall", "mid", "small"]

TILE_WIDTH = 50
BUFFER = 25
MIN_CHECKPOINT_MB = 400
OVERWRITE_MODELS = False

shutil.rmtree(WORK, ignore_errors=True)

for directory in (STACKS, CROPS, TILES, KEEP, RUNS):
    directory.mkdir(parents=True, exist_ok=True)

paths = {
    site: {
        "rgb": ROOT / site / f"{site}_RGB.TIF",
        "ned": ROOT / site / f"{site}_NED.TIF",
        "stack": STACKS / f"{site}_h1_8bit_6band.tif",
        "mask": ROOT / site / f"mask_{site}.gpkg",
        "aoi": {
            stratum: ROOT / site / f"{site}_aoi_{stratum}.gpkg"
            for stratum in STRATA
        },
        "crowns": {
            stratum: ROOT / site / f"crowns_{site}_{stratum}.gpkg"
            for stratum in STRATA
        },
    }
    for site in SITES
}

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Input validation

All required files are located before preprocessing. Paired RGB and NED products are checked for consistent dimensions, spatial alignment, coordinate reference system, resolution and datatype.

In [ ]:
required_files = []

for site in SITES:
    required_files.extend([
        paths[site]["rgb"],
        paths[site]["ned"],
        paths[site]["mask"],
        *paths[site]["aoi"].values(),
        *paths[site]["crowns"].values(),
    ])

missing_files = [
    path for path in required_files
    if not path.exists()
]

assert not missing_files, (
    "Missing source files:\n"
    + "\n".join(map(str, missing_files))
)

for site in SITES:
    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        assert rgb.count >= 3 and ned.count >= 3
        assert rgb.crs == ned.crs, f"{site}: CRS mismatch"
        assert rgb.shape == ned.shape, f"{site}: raster shape mismatch"
        assert rgb.transform == ned.transform, f"{site}: transform mismatch"
        assert rgb.bounds == ned.bounds, f"{site}: bounds mismatch"
        assert rgb.res == ned.res, f"{site}: resolution mismatch"
        assert rgb.dtypes[:3] == ned.dtypes[:3], f"{site}: datatype mismatch"
        assert set(rgb.dtypes[:3]) == {"uint16"}, (
            f"{site}: expected uint16 source imagery"
        )

        print(
            f"{site}: {rgb.width} × {rgb.height} pixels, "
            f"{rgb.res[0]:.3f} m resolution, {rgb.crs}"
        )

## RGB preprocessing

To reproduce the original preprocessing, the paired products are combined into a six-band working raster, although the H1 model reads only the first three RGB bands. Values are converted from 16-bit to 8-bit using site- and band-specific 2nd and 98th percentiles. Valid pixels are scaled to 1–254, with 0 reserved for nodata.

In [ ]:
BAND_NAMES = [
    "red",
    "green",
    "blue",
    "near-infrared",
    "red-edge",
    "deep-blue",
]


def build_h1_stack(site):
    output_path = paths[site]["stack"]

    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        histograms = np.zeros((6, 65536), dtype=np.int64)

        for _, window in rgb.block_windows(1):
            values = np.concatenate([
                rgb.read([1, 2, 3], window=window),
                ned.read([1, 2, 3], window=window),
            ], axis=0)

            valid = values.sum(axis=0) > 0

            for band in range(6):
                histograms[band] += np.bincount(
                    values[band][valid].ravel().astype(np.int64),
                    minlength=65536,
                )

        lower = np.empty(6, dtype=np.float32)
        upper = np.empty(6, dtype=np.float32)

        for band in range(6):
            histograms[band, 0] = 0
            cumulative = np.cumsum(histograms[band])

            assert cumulative[-1] > 0, (
                f"{site}/{BAND_NAMES[band]}: no valid pixels"
            )

            lower[band] = np.searchsorted(
                cumulative,
                0.02 * cumulative[-1],
            )
            upper[band] = np.searchsorted(
                cumulative,
                0.98 * cumulative[-1],
            )

            assert upper[band] > lower[band], (
                f"{site}/{BAND_NAMES[band]}: invalid stretch limits"
            )

        metadata = rgb.meta.copy()
        metadata.update(
            driver="GTiff",
            count=6,
            dtype="uint8",
            nodata=0,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            compress="deflate",
        )

        lower_3d = lower.reshape(-1, 1, 1)
        upper_3d = upper.reshape(-1, 1, 1)

        with rasterio.open(output_path, "w", **metadata) as destination:
            for _, window in rgb.block_windows(1):
                values = np.concatenate([
                    rgb.read([1, 2, 3], window=window),
                    ned.read([1, 2, 3], window=window),
                ], axis=0).astype(np.float32)

                nodata = values.sum(axis=0) == 0

                image = (
                    (values - lower_3d)
                    / (upper_3d - lower_3d)
                    * 253
                    + 1
                )
                image = np.clip(image, 1, 254)
                image[np.broadcast_to(nodata, image.shape)] = 0

                destination.write(
                    image.astype(np.uint8),
                    window=window,
                )

    with rasterio.open(output_path) as raster:
        assert raster.count == 6
        assert raster.dtypes == ("uint8",) * 6
        assert raster.nodata == 0

    limits = ", ".join(
        f"{name}: {int(low)}–{int(high)}"
        for name, low, high in zip(
            BAND_NAMES,
            lower,
            upper,
        )
    )

    print(f"{site}: {limits}")
    return output_path


for site in SITES:
    build_h1_stack(site)

## Valid regions

Boundary-crown masks are subtracted from each AOI to define the region retained during tiling and evaluation. Invalid geometries are repaired using a zero-width buffer, and crown centroids are checked against their corresponding AOI.

In [ ]:
prepped = {}

for site in SITES:
    with rasterio.open(paths[site]["stack"]) as raster:
        raster_crs = raster.crs

    boundary_mask = gpd.read_file(
        paths[site]["mask"]
    ).to_crs(raster_crs)
    boundary_mask.geometry = boundary_mask.geometry.buffer(0)

    prepped[site] = {}

    for stratum in STRATA:
        aoi = gpd.read_file(
            paths[site]["aoi"][stratum]
        ).to_crs(raster_crs)

        crowns = gpd.read_file(
            paths[site]["crowns"][stratum]
        ).to_crs(raster_crs)

        aoi.geometry = aoi.geometry.buffer(0)
        crowns.geometry = crowns.geometry.buffer(0)

        assert aoi.geometry.is_valid.all()
        assert crowns.geometry.is_valid.all()

        inside_aoi = crowns.geometry.centroid.within(
            aoi.union_all()
        )

        assert inside_aoi.all(), (
            f"{site}/{stratum}: "
            f"{int((~inside_aoi).sum())} crown centroids outside AOI"
        )

        keep = gpd.overlay(
            aoi,
            boundary_mask,
            how="difference",
        )
        keep.geometry = keep.geometry.buffer(0)

        assert len(keep) > 0
        assert keep.geometry.is_valid.all()

        keep_path = KEEP / f"keep_{site}_{stratum}.gpkg"
        keep.to_file(keep_path, driver="GPKG")

        prepped[site][stratum] = {
            "crowns": crowns,
            "keep_path": keep_path,
        }

        valid_area_ha = keep.area.sum() / 10_000

        print(
            f"{site}/{stratum}: "
            f"{len(crowns)} crowns, "
            f"{valid_area_ha:.2f} valid ha"
        )

## Image tiling

Each 300 m AOI is cropped with a 2 m square buffer and divided into overlapping 100 m tiles. Detectree2 uses a 50 m core with a 25 m buffer on each side, producing a 50 m step. No crown-coverage threshold is imposed, and additional RGB contrast enhancement is disabled.

In [ ]:
from rasterio.mask import mask as rio_mask
from detectree2.preprocessing.tiling import tile_data


def tile_aoi(site, stratum):
    crowns = prepped[site][stratum]["crowns"]
    keep_path = prepped[site][stratum]["keep_path"]

    aoi = gpd.read_file(
        paths[site]["aoi"][stratum]
    ).to_crs(crowns.crs)

    crop_path = CROPS / f"{site}_{stratum}_h1.tif"

    with rasterio.open(paths[site]["stack"]) as source:
        image, transform = rio_mask(
            source,
            aoi.geometry.buffer(2, join_style=2),
            crop=True,
            nodata=0,
        )

        metadata = source.meta.copy()
        metadata.update(
            height=image.shape[1],
            width=image.shape[2],
            transform=transform,
            nodata=0,
        )

    with rasterio.open(crop_path, "w", **metadata) as destination:
        destination.write(image)

    output_directory = (
        TILES
        / f"{site}_{stratum}_rgb_{TILE_WIDTH}_{BUFFER}"
    )

    tile_data(
        img_path=str(crop_path),
        out_dir=str(output_directory),
        buffer=BUFFER,
        tile_width=TILE_WIDTH,
        tile_height=TILE_WIDTH,
        crowns=crowns,
        threshold=0.0,
        nan_threshold=1.0,
        full_coverage=False,
        mode="rgb",
        mask_path=str(keep_path),
        use_convex_mask=False,
        enhance_rgb_contrast=False,
        tile_placement="grid",
        multithreaded=True,
        ignore_bands_indices=[],
    )

    tile_count = len(
        list(output_directory.glob("*.geojson"))
    )

    assert tile_count == 25, (
        f"{site}/{stratum}: "
        f"expected 25 tiles, found {tile_count}"
    )

    print(f"{site}/{stratum}: {tile_count} tiles")
    return output_directory


rgb_directories = {
    (site, stratum): tile_aoi(site, stratum)
    for site in SITES
    for stratum in STRATA
}

total_tiles = sum(
    len(list(directory.glob("*.geojson")))
    for directory in rgb_directories.values()
)

assert total_tiles == 225
print("Total tiles:", total_tiles)

## Leave-one-site-out partitions

Each fold withholds one complete site for testing. At the two development sites, high- and low-canopy AOIs are used for training and intermediate-canopy AOIs are reserved for validation. This produces 100 training, 50 validation and 75 test tiles per fold.

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectree2.models.train import get_tree_dicts


def register_fold(
    directories,
    holdout,
    tag,
    validation_stratum="mid",
):
    dataset_names = {
        split: f"{tag}_{split}"
        for split in ("train", "val", "test")
    }

    for name in dataset_names.values():
        if name in DatasetCatalog:
            DatasetCatalog.remove(name)
            MetadataCatalog.remove(name)

    partitions = {
        "train": [],
        "val": [],
        "test": [],
    }

    for (site, stratum), directory in directories.items():
        records = get_tree_dicts(str(directory))

        if site == holdout:
            partitions["test"].extend(records)
        elif stratum == validation_stratum:
            partitions["val"].extend(records)
        else:
            partitions["train"].extend(records)

    for split, records in partitions.items():
        name = dataset_names[split]

        DatasetCatalog.register(
            name,
            lambda records=records: records,
        )
        MetadataCatalog.get(name).set(
            thing_classes=["tree"]
        )

    counts = tuple(
        len(partitions[split])
        for split in ("train", "val", "test")
    )

    assert counts == (100, 50, 75), (
        f"{tag}: expected 100/50/75 tiles, found {counts}"
    )

    print(
        f"{tag}: train {counts[0]} | "
        f"validation {counts[1]} | "
        f"test {counts[2]}"
    )

    return (
        dataset_names["train"],
        dataset_names["val"],
        dataset_names["test"],
    )


for holdout in SITES:
    register_fold(
        rgb_directories,
        holdout,
        f"h1_check_{holdout}",
    )

## Model configuration

A ResNet-101 FPN Mask R-CNN is initialized from Detectree2’s multisite tropical-forest checkpoint. The backbone remains unfrozen. Training uses a batch size of two, a base learning rate of 3.389 × 10⁻⁴, random resizing and a maximum of 3,000 iterations. Validation is performed every 100 iterations, with early stopping after five evaluations without improvement.

In [ ]:
from detectron2.data import build_detection_test_loader
import detectron2.data.transforms as T

from detectree2.models.train import (
    FlexibleDatasetMapper,
    MyTrainer,
    setup_cfg,
)


required_parameters = {
    "imgmode",
    "num_bands",
    "resize",
    "eval_period",
}

assert required_parameters.issubset(
    inspect.signature(setup_cfg).parameters
), "Unexpected Detectree2 API"


def fixed_validation_loader(cls, cfg, dataset_name):
    mapper = FlexibleDatasetMapper(
        cfg,
        is_train=False,
        augmentations=[
            T.ResizeShortestEdge(
                [1000, 1000],
                1333,
            )
        ],
    )

    return build_detection_test_loader(
        cfg,
        dataset_name,
        mapper=mapper,
    )


MyTrainer.build_test_loader = classmethod(
    fixed_validation_loader
)

PRETRAINED_MODEL = Path(
    "/content/230103_randresize_full.pth"
)

if not PRETRAINED_MODEL.exists():
    urllib.request.urlretrieve(
        "https://zenodo.org/records/10522461/"
        "files/230103_randresize_full.pth",
        PRETRAINED_MODEL,
    )

assert PRETRAINED_MODEL.stat().st_size / 1e6 > 400, (
    "Pretrained checkpoint is incomplete"
)

## Model training

One model is trained for each held-out site. Existing complete checkpoints are retained unless `OVERWRITE_MODELS` is enabled.

In [ ]:
def valid_checkpoint(path):
    return (
        path.exists()
        and path.stat().st_size / 1e6 >= MIN_CHECKPOINT_MB
    )


def train_fold(holdout):
    tag = f"rgb_holdout_{holdout}"
    output_directory = RUNS / tag
    best_checkpoint = output_directory / "model_best.pth"

    if (
        valid_checkpoint(best_checkpoint)
        and not OVERWRITE_MODELS
    ):
        print(f"{tag}: retaining existing model_best.pth")
        return output_directory, False

    if output_directory.exists():
        shutil.rmtree(output_directory)

    train_name, validation_name, _ = register_fold(
        rgb_directories,
        holdout,
        tag,
    )

    cfg = setup_cfg(
        base_model=(
            "COCO-InstanceSegmentation/"
            "mask_rcnn_R_101_FPN_3x.yaml"
        ),
        trains=(train_name,),
        tests=(validation_name,),
        update_model=str(PRETRAINED_MODEL),
        workers=2,
        ims_per_batch=2,
        base_lr=0.0003389,
        backbone_freeze=0,
        max_iter=3000,
        eval_period=100,
        resize="rand_fixed",
        imgmode="rgb",
        num_bands=3,
        out_dir=str(output_directory),
    )

    cfg.SOLVER.CHECKPOINT_PERIOD = 500

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )
    (output_directory / "config.yaml").write_text(
        cfg.dump()
    )

    trainer = MyTrainer(cfg, patience=5)
    trainer.resume_or_load(resume=False)
    trainer.train()

    return output_directory, True

## Checkpoint selection

Detectree2 saves a numbered checkpoint whenever validation AP50 reaches a new maximum. The checkpoint corresponding to the highest recorded segmentation AP50 is retained as `model_best.pth`.

In [ ]:
def validation_history(output_directory):
    metrics_path = Path(output_directory) / "metrics.json"

    assert metrics_path.exists(), (
        f"Missing metrics file: {metrics_path}"
    )

    records = [
        json.loads(line)
        for line in metrics_path.read_text().splitlines()
    ]

    evaluations = [
        (record["iteration"], record["segm/AP50"])
        for record in records
        if "segm/AP50" in record
    ]

    assert evaluations, (
        f"No segmentation AP50 values in {metrics_path}"
    )

    return evaluations


def promote_best_checkpoint(output_directory):
    output_directory = Path(output_directory)
    evaluations = validation_history(output_directory)

    best_position = max(
        range(len(evaluations)),
        key=lambda index: evaluations[index][1],
    )

    best_iteration, best_ap50 = evaluations[best_position]
    source = output_directory / f"model_{best_position + 1}.pth"
    destination = output_directory / "model_best.pth"

    available = [
        path.name
        for path in sorted(output_directory.glob("model_*.pth"))
    ]

    assert source.exists(), (
        f"{output_directory.name}: expected {source.name}; "
        f"available checkpoints: {available}"
    )

    shutil.copy2(source, destination)

    assert valid_checkpoint(destination), (
        f"Incomplete checkpoint: {destination}"
    )

    print(
        f"{output_directory.name}: "
        f"{destination.name} <- {source.name}; "
        f"iteration {best_iteration}, "
        f"AP50 {best_ap50:.2f}"
    )

    return best_iteration, best_ap50

## Training execution

The three folds are processed sequentially. Newly trained checkpoints are selected and synchronized before the next fold begins.

In [ ]:
training_summary = []

for holdout in ["mbeli", "dzanga", "lokoue"]:
    print(f"\n===== Hold out {holdout} =====")

    output_directory, trained = train_fold(holdout)

    if trained:
        best_iteration, best_ap50 = (
            promote_best_checkpoint(output_directory)
        )
        os.sync()
    else:
        evaluations = validation_history(
            output_directory
        )
        best_iteration, best_ap50 = max(
            evaluations,
            key=lambda result: result[1],
        )

    training_summary.append({
        "holdout": holdout,
        "output_directory": str(output_directory),
        "trained_in_current_run": trained,
        "best_iteration": best_iteration,
        "best_validation_ap50": best_ap50,
    })

print("\nTraining complete.")

## Output verification

Each final checkpoint is checked for completeness and the expected three-channel model input. A summary of the selected validation results is saved alongside the trained models.

In [ ]:
for result in training_summary:
    checkpoint = (
        Path(result["output_directory"])
        / "model_best.pth"
    )

    assert valid_checkpoint(checkpoint), (
        f"Missing or incomplete checkpoint: {checkpoint}"
    )

    state = torch.load(
        checkpoint,
        map_location="cpu",
    )

    conv1_weights = state["model"][
        "backbone.bottom_up.stem.conv1.weight"
    ]

    assert conv1_weights.shape[1] == 3, (
        f"{checkpoint}: expected three input channels"
    )

    print(
        f"{result['holdout']}: "
        f"conv1 {tuple(conv1_weights.shape)}, "
        f"best AP50 {result['best_validation_ap50']:.2f} "
        f"at iteration {result['best_iteration']}"
    )

summary_path = RUNS / "h1_training_summary.json"
summary_path.write_text(
    json.dumps(training_summary, indent=2)
)

print("Saved:", summary_path)